In [ ]:
from pystac.client import Client
from odc.stac import load
import xarray as xr
from utils import mask_land, mask_deeps, make_indices, do_prediction, locations
import joblib
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

from ipyleaflet import basemaps
import folium

import warnings
warnings.filterwarnings("ignore")

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
collection = "sentinel-2-l2a"

In [ ]:
location = locations.suva
daterange = "2024"

In [ ]:
# Consider removing very cloudy scenes...
items = catalog.search(
    collections=[collection],
    bbox=location.bbox,
    datetime=daterange,
    query={"eo:cloud_cover": {"lt": 50}},
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
data = load(
    items,
    bbox=location.bbox,
    epsg="utm",
    measurements=[
        "scl",
        "nir",
        "red",
        "blue",
        "green",
        "nir08",
        "nir09",
        "swir16",
        "swir22",
        "coastal",
        "rededge1",
        "rededge2",
        "rededge3",
    ],
    chunks={"x": 2048, "y": 2048},
    nodata=0,
    groupby="solar_day"
)

# Mask clouds
mask = data.scl.isin([3, 8, 9, 10])
data = data.where(~mask)
data = make_indices(data)

# Mask land
data = mask_land(data)

# # Mask deep water
data = mask_deeps(data)

data = data.drop_vars("scl")

data

In [ ]:
model = joblib.load("models/2025_03_11_randomforest_both_masks_30m.joblib")

def predict_for_day(day):
    return do_prediction(data.sel(time=day), model).compute()

with ThreadPoolExecutor() as executor:
    predictions_list = list(tqdm(executor.map(predict_for_day, data.time), total=len(data.time)))

# Concatenate them all together again
predictions = xr.concat(predictions_list, dim="time").to_dataset(name="elevation")

predictions

In [ ]:
# Plot all the timesteps
predictions.elevation.plot.imshow(col="time", col_wrap=2, cmap="Blues_r", robust=True, size=6)

In [ ]:
total * 0.15

In [ ]:
# Clean up the data by removing pixels that only had predictions sometimes
count = predictions.elevation.count(dim="time")
total = len(predictions.time)

mask = count > (total * 0.15)  # At least X% of the time there was a prediction

mean = predictions.elevation.mean(dim="time")
stdev = predictions.elevation.std(dim="time")

stdev_mask = stdev < 3

mean_countmasked = mean.where(mask)
mean_stdevmasked = mean.where(stdev_mask)

# Make a fancy map
centroid = list(predictions.odc.geobox.geographic_extent.centroid.coords[0])[::-1]
m = folium.Map(location=centroid, zoom_start=12)
_ = folium.TileLayer(tiles=basemaps.Esri.WorldImagery).add_to(m)

count.odc.add_to(m, cmap="Reds", name="Count")
stdev.odc.add_to(m, cmap="Reds", name="Stdev")

mean.odc.add_to(m, cmap="Blues_r", name="Depth")

mean_stdevmasked.odc.add_to(m, cmap="Blues_r", name="Depth (stdev masked)")
mean_countmasked.odc.add_to(m, cmap="Blues_r", name="Depth (count masked)")

folium.LayerControl().add_to(m)

m

In [ ]:
count.plot.imshow()

In [ ]:
mean_countmasked.odc.write_cog(f"{location.name}_deep_masked_30m.tif", overwrite=True)